# Conexión a la API de ENTSO-E

**Guía del equipo GridForecast** · Autor: Jalil Muñoz Robles

Este notebook explica, paso a paso, cómo nos conectamos a la **ENTSO-E Transparency Platform** para descargar datos de la red eléctrica europea: qué librerías usamos, cómo autenticarnos de forma segura y cómo obtener información específica (demanda, generación, pronósticos renovables y precios).

> Es material de aprendizaje y referencia. El script de producción que automatiza este flujo vive en `src/ingest_load.py`.

## 1. Las librerías que usamos

Tres piezas, cada una con un trabajo claro:

- **`entsoe-py`** — un cliente de Python que envuelve la API REST de ENTSO-E. Le pides datos y te los devuelve directamente como objetos de pandas, sin que tengas que interpretar el XML crudo de la API. Usamos su clase `EntsoePandasClient`.
- **`pandas`** — la librería estándar para datos tabulares (DataFrames) y series de tiempo. ENTSO-E devuelve series indexadas por fecha/hora, y pandas es la herramienta natural para trabajarlas.
- **`python-dotenv`** — carga variables de entorno desde un archivo `.env` local. La usamos para leer el token sin escribirlo nunca en el código.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from entsoe import EntsoePandasClient

print("Librerías importadas correctamente")
print("Versión de pandas:", pd.__version__)

## 2. Cargar el token de forma segura

El acceso a la API requiere un **token** (una especie de contraseña). La regla de oro: **el token nunca se escribe en el código ni se sube a GitHub.** Vive en un archivo `.env` en la raíz del proyecto, protegido por `.gitignore`.

`find_dotenv()` busca el archivo `.env` subiendo desde la carpeta actual, así que funciona aunque ejecutes el notebook desde `notebooks/`.

In [ ]:
load_dotenv(find_dotenv())
token = os.getenv("ENTSOE_API_TOKEN")

if not token or token == "your_token_here":
    raise SystemExit(
        "No se encontró el token. Copia .env.example a .env y pega tu token de ENTSO-E."
    )

# Nunca imprimimos el token, ni siquiera parcialmente.
print("Token cargado correctamente")

## 3. Crear el cliente

Con el token, creamos el objeto `EntsoePandasClient`. Este cliente es nuestra "puerta" a todos los datos: cada consulta será un método de este objeto.

In [ ]:
client = EntsoePandasClient(api_key=token)
print("Cliente de ENTSO-E listo")

## 4. Fechas con zona horaria (un detalle que importa)

ENTSO-E **exige** marcas de tiempo con zona horaria (*timezone-aware*). No basta con "1 de enero"; hay que decir "1 de enero en horario de Europa/Madrid". Esto evita ambigüedades con los cambios de horario y entre husos.

Definimos un rango de los últimos 7 días. `pd.Timestamp.now(tz=...)` da el momento actual con zona horaria, y restamos una ventana con `pd.Timedelta`.

In [ ]:
TIMEZONE = "Europe/Madrid"

end = pd.Timestamp.now(tz=TIMEZONE)
start = end - pd.Timedelta(days=7)

print("Desde:", start)
print("Hasta:", end)

## 5. Obtener información específica

Aquí está el corazón: cada tipo de dato es un método distinto del cliente. Todos reciben el **código de país** y el **rango de fechas**. Veamos los cuatro más útiles para GridForecast.

### 5.1 Demanda eléctrica (carga real)

Es nuestra variable objetivo principal: cuánta electricidad consumió el sistema. `query_load` devuelve la demanda real medida.

In [ ]:
demanda = client.query_load("ES", start=start, end=end)
demanda.head()

### 5.2 Generación por tipo de tecnología

Cuánta electricidad se generó y con qué fuente (gas, nuclear, hidráulica, eólica, solar...). Con `psr_type=None` devuelve todas las tecnologías.

In [ ]:
generacion = client.query_generation("ES", start=start, end=end, psr_type=None)
generacion.head()

### 5.3 Pronóstico de eólica y solar

El pronóstico oficial de generación renovable. Es clave para el proyecto: parte de lo que queremos predecir y contrastar.

In [ ]:
renovables = client.query_wind_and_solar_forecast("ES", start=start, end=end, psr_type=None)
renovables.head()

### 5.4 Precios del mercado diario (day-ahead)

El precio mayorista de la electricidad por hora. Enriquece el análisis y puede servir como variable explicativa.

In [ ]:
precios = client.query_day_ahead_prices("ES", start=start, end=end)
precios.head()

## 6. Cambiar de país

ENTSO-E cubre decenas de países; solo hay que cambiar el código. Algunos ejemplos:

- `"ES"` — España
- `"PT"` — Portugal
- `"FR"` — Francia
- `"DE_LU"` — Alemania-Luxemburgo
- `"IT"` — Italia

Esto permite escalar el proyecto (por ejemplo, comparar el comportamiento de la red entre países) sin cambiar de fuente de datos.

In [ ]:
# Ejemplo: demanda de Portugal en el mismo rango
demanda_pt = client.query_load("PT", start=start, end=end)
demanda_pt.head()

## 7. Explorar y guardar

Con los datos en un DataFrame de pandas, explorarlos y guardarlos es directo.

In [ ]:
print("Forma (filas, columnas):", demanda.shape)
print("Rango de fechas:", demanda.index.min(), "->", demanda.index.max())
demanda.describe()

In [ ]:
# Guardar en Parquet (formato eficiente y versionable).
# La carpeta data/ está en .gitignore: los datos NO se suben al repo.
from pathlib import Path

Path("../data").mkdir(exist_ok=True)
demanda.to_parquet("../data/es_load_ejemplo.parquet")
print("Guardado en data/es_load_ejemplo.parquet")

## 8. Buenas prácticas y notas

- **Nunca subas el `.env`.** El token es personal; cada integrante genera el suyo.
- **Uso justo:** la API es un recurso compartido. Evita descargar rangos enormes en bucle; pide solo lo que necesitas.
- **Atribución:** los datos son de ENTSO-E (Reglamento UE 543/2013). Hay que citar a ENTSO-E como fuente en el proyecto.
- **Zona horaria:** siempre usa timestamps con zona; es la causa más común de errores en las consultas.

---

Con esto tienes el mapa completo de cómo GridForecast obtiene sus datos. El script `src/ingest_load.py` automatiza este flujo para la demanda; los demás métodos que viste aquí son los siguientes ladrillos del pipeline.